
---

### Step 1: โหลด Dataset จาก Kaggle

ก่อนอื่นต้องเตรียมไฟล์ `kaggle.json` (จาก Account Setting ใน Kaggle -> Create New Token) รันเซลล์นี้เพื่ออัปโหลดไฟล์ และดาวน์โหลดข้อมูล

In [ ]:
# Cell 1: Setup Kaggle API & Download Data
!pip install -q kaggle
from google.colab import files

print("กรุณาอัปโหลดไฟล์ kaggle.json ของคุณ")
files.upload() # กดปุ่ม Choose Files เพื่ออัปโหลด kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ดาวน์โหลดและแตกไฟล์ Dataset (อาจใช้เวลาสักครู่สำหรับ 1.35 GB)
!kaggle competitions download -c super-ai-engineer-season-6-individual-hackathon-house-recognition
!mkdir -p dataset
!unzip -q super-ai-engineer-season-6-individual-hackathon-house-recognition.zip -d dataset
print("Extract เสร็จสิ้น!")

กรุณาอัปโหลดไฟล์ kaggle.json ของคุณ


Saving kaggle.json to kaggle.json
100% 1.25G/1.25G [00:08<00:00, 158MB/s] 

Extract เสร็จสิ้น!


---

### Step 2: สร้างและเทรน Model (PyTorch รับจบ)

โค้ดส่วนนี้จะอ่านไฟล์ `train.csv` โหลดรูป เทรนด้วยโมเดล **ResNet50** (หรือเปลี่ยนเป็นตัวอื่นได้) และเตรียมพร้อมสำหรับ Prediction

In [ ]:
# First, let's fix the directory path issue.
# I will add a check to find the correct image directory.
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

# ==========================================
# 1. Configuration & Hyperparameters
# ==========================================
# Let's verify the path. Often unzip creates dataset/train/train/...
# If dataset/train/train exists, we use that.
if os.path.exists('dataset/train/train'):
    TRAIN_IMG_DIR = 'dataset/train/train'
else:
    TRAIN_IMG_DIR = 'dataset/train'

TRAIN_CSV = 'dataset/train.csv'
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
print(f"Using image directory: {TRAIN_IMG_DIR}")

train_df = pd.read_csv(TRAIN_CSV)
IMAGE_COL = 'image_name'
LABEL_COL = 'class'

labels = sorted(train_df[LABEL_COL].unique())
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
train_df['label_idx'] = train_df[LABEL_COL].map(label_to_idx)
NUM_CLASSES = len(labels)

# ==========================================
# 2. Dataset & Transforms
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class HouseDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = str(self.dataframe.iloc[idx][IMAGE_COL])
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'

        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Fallback check if the file is directly in dataset/train
            fallback_path = os.path.join('dataset/train', img_name)
            image = Image.open(fallback_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = self.dataframe.iloc[idx]['label_idx']
        return image, label

train_dataset = HouseDataset(train_df, TRAIN_IMG_DIR, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, targets in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        pbar.set_postfix({'Loss': running_loss/total, 'Acc': correct/total})

print("Training Complete!")

Using device: cuda
Using image directory: dataset/train/train


Epoch 1/5:   0%|          | 0/93 [00:00<?, ?it/s]

Epoch 2/5:   0%|          | 0/93 [00:00<?, ?it/s]

Epoch 3/5:   0%|          | 0/93 [00:00<?, ?it/s]

Epoch 4/5:   0%|          | 0/93 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5a62bfec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5a62bfec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch 5/5:   0%|          | 0/93 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5a62bfec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7af5a62bfec0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Training Complete!


---

### Step 3: ทำนายผล (Inference) และสร้างไฟล์ Submission

รันเซลล์สุดท้ายนี้เพื่อดึงรูปจากโฟลเดอร์ `test` มาทำนาย และสร้างไฟล์ `submission.csv` เพื่อนำไปอัปโหลดขึ้น Kaggle ครับ

In [ ]:
# Cell 3: Inference & Submission
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

# Check for nested test directory
if os.path.exists('dataset/test/test'):
    TEST_IMG_DIR = 'dataset/test/test'
else:
    TEST_IMG_DIR = 'dataset/test'

SAMPLE_SUB_CSV = 'dataset/sample_submission.csv'
sub_df = pd.read_csv(SAMPLE_SUB_CSV)
TEST_IMAGE_COL = 'id'

print(f"Using test image directory: {TEST_IMG_DIR}")

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class TestDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = str(self.dataframe.iloc[idx][TEST_IMAGE_COL])
        if not img_name.endswith('.jpg'):
            img_name += '.jpg'

        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Fallback if nested structure is different
            fallback_path = os.path.join('dataset/test', img_name)
            image = Image.open(fallback_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name

test_dataset = TestDataset(sub_df, TEST_IMG_DIR, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
predictions = []

print("Predicting test data...")
with torch.no_grad():
    for images, _ in tqdm(test_loader):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())

final_answers = [idx_to_label[p] for p in predictions]
TARGET_COL = 'answer' if 'answer' in sub_df.columns else 'class'
sub_df[TARGET_COL] = final_answers

sub_df.to_csv('submission.csv', index=False)
print(f"Saved to submission.csv! Column '{TARGET_COL}' updated.")

Using test image directory: dataset/test/test
Predicting test data...


  0%|          | 0/49 [00:00<?, ?it/s]

Saved to submission.csv! Column 'answer' updated.


### 💡 ทริคแนะนำเพิ่มเติมเพื่อดันคะแนนให้อยู่หัวตาราง:

1. **เปลี่ยน Model:** ในหัวข้อ Model Definition ลองเปลี่ยน `resnet50` เป็น `efficientnet_b2` หรือ `convnext_tiny` จะให้ผลลัพธ์ที่ดีกว่าสำหรับภาพที่มีความซับซ้อน
2. **Epochs:** 5 Epochs อาจจะน้อยไปสำหรับรับจบ ลองปรับเพิ่มเป็น `15-20` Epochs เพื่อให้โมเดลลู่เข้า (Converge) มากขึ้น
3. **ตรวจสอบชื่อคอลัมน์:** หาก Error ตอนอ่าน Dataset ให้ลอง `print(train_df.columns)` เพื่อเช็คดูว่าชื่อคอลัมน์มันคือ `id`, `answer` จริงๆ หรือเป็นคำอื่น แล้วปรับตัวแปร `IMAGE_COL` กับ `LABEL_COL` ให้ตรงครับ

ขอให้โชคดีในการแข่ง Super AI Engineer Season 6 ครับ ลุยเลย! 🚀